# Bird Species Classification with ResNet-18

Standalone PyTorch notebook for training, validating, testing, and predicting ten bird species. All required dataset, model, training, evaluation, and inference code is contained in this notebook.

## 1. Install compatible dependencies

Run this cell once, then restart the kernel before running the remaining cells. This notebook deliberately does not use `datasets` or `pyarrow`, avoiding the dataset-script and PyArrow/NumPy issues.

In [ ]:
import sys
import subprocess

core_packages = [
    'numpy==2.2.6',
    'pandas>=2.2.3',
    'matplotlib>=3.9',
    'seaborn>=0.13',
    'scikit-learn>=1.5',
    'Pillow>=10.4',
    'packaging>=24.0',
    'tqdm>=4.66',
    'huggingface_hub>=0.25'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--no-cache-dir', *core_packages])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--no-cache-dir',
    'torch==2.6.0', 'torchvision==0.21.0',
    '--index-url', 'https://download.pytorch.org/whl/cpu'
])
print('Installation completed. Restart the Jupyter kernel, then continue with the next cell.')

## 2. Verify the runtime

This cell checks the NumPy, PyTorch, and torchvision versions and confirms that PyTorch can exchange arrays with NumPy.

In [ ]:
from importlib.metadata import version
from packaging.version import Version
import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib
import seaborn as sns
import sklearn
from PIL import Image

versions = {
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'scikit-learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': sns.__version__,
    'Pillow': version('Pillow')
}
print(versions)
assert Version(np.__version__) >= Version('2.0.0'), 'Run the installation cell and restart the kernel so NumPy 2.x is active.'
assert Version(torch.__version__.split('+')[0]) == Version('2.6.0'), 'Run the installation cell and restart the kernel so the compatible PyTorch build is active.'
assert Version(torchvision.__version__.split('+')[0]) == Version('0.21.0'), 'Run the installation cell and restart the kernel so the compatible torchvision build is active.'
assert torch.tensor([1.0, 2.0]).numpy().shape == (2,)
assert torchvision.ops.nms(torch.tensor([[0., 0., 1., 1.]]), torch.tensor([1.0]), 0.5).tolist() == [0]
print('Runtime verification passed.')

## 3. Imports, configuration, and reproducibility

In [ ]:
import os
import io
import json
import random
import tarfile
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
MAX_EPOCHS = 6
HEAD_ONLY_EPOCHS = 2
PATIENCE = 3
NUM_WORKERS = 0
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_BACKBONE = 1e-4
LEARNING_RATE_CLASSIFIER = 5e-4
WEIGHT_DECAY = 1e-4
DROPOUT = 0.25
LABEL_SMOOTHING = 0.05

SELECTED_SPECIES = [
    'ABBOTTS BABBLER',
    'ABBOTTS BOOBY',
    'ABYSSINIAN GROUND HORNBILL',
    'AFRICAN CROWNED CRANE',
    'AFRICAN EMERALD CUCKOO',
    'AFRICAN FIREFINCH',
    'AFRICAN OYSTER CATCHER',
    'AFRICAN PIED HORNBILL',
    'AFRICAN PYGMY GOOSE',
    'ALBATROSS'
]

CANDIDATE_DATA_DIRS = [
    Path(r'C:\Users\vivan\.gemini\antigravity-ide\scratch\bird-species-classifier\data\processed'),
    Path.cwd() / 'data' / 'processed',
    Path.cwd() / 'bird_species_data' / 'processed'
]
DATA_DIR = next((path for path in CANDIDATE_DATA_DIRS if path.exists()), Path.cwd() / 'bird_species_data' / 'processed')
ARTIFACT_DIR = Path.cwd() / 'bird_species_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACT_DIR / 'resnet18_best.pt'
HISTORY_PATH = ARTIFACT_DIR / 'resnet18_history.csv'
METRICS_PATH = ARTIFACT_DIR / 'resnet18_test_metrics.json'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_to_idx = {species: index for index, species in enumerate(SELECTED_SPECIES)}
idx_to_class = {index: species for species, index in class_to_idx.items()}
print(f'Device: {device}')
print(f'Data directory: {DATA_DIR}')
print(f'Artifacts directory: {ARTIFACT_DIR}')

## 4. Prepare the dataset

If the processed folders already exist, the notebook uses them. Otherwise it downloads the public dataset archives from Hugging Face and extracts the ten selected species. The fallback download is large because the training archive is about 1.87 GB.

In [ ]:
def split_is_ready(root):
    return all((root / split / species).is_dir() for split in ['train', 'val', 'test'] for species in SELECTED_SPECIES)

def extract_selected_species():
    archive_map = {'train': 'data/train.tar.gz', 'val': 'data/valid.tar.gz', 'test': 'data/test.tar.gz'}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    selected_lookup = {species.casefold(): species for species in SELECTED_SPECIES}
    for target_split, filename in archive_map.items():
        archive_path = hf_hub_download(
            repo_id='chriamue/bird-species-dataset',
            repo_type='dataset',
            filename=filename
        )
        extracted = 0
        with tarfile.open(archive_path, mode='r:gz') as archive:
            for member in tqdm(archive, desc=f'Extracting {target_split}'):
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) < 2:
                    continue
                candidate = parts[-2].replace('_', ' ').replace('-', ' ').casefold()
                if candidate not in selected_lookup:
                    continue
                species = selected_lookup[candidate]
                destination = DATA_DIR / target_split / species / Path(parts[-1]).name
                destination.parent.mkdir(parents=True, exist_ok=True)
                if not destination.exists():
                    source = archive.extractfile(member)
                    if source is not None:
                        with source, destination.open('wb') as target:
                            shutil.copyfileobj(source, target)
                extracted += 1
        print(f'{target_split}: {extracted} matching image files available')

if not split_is_ready(DATA_DIR):
    extract_selected_species()

if not split_is_ready(DATA_DIR):
    raise FileNotFoundError(f'Expected train, val, and test folders for every selected species under {DATA_DIR}.')

split_counts = {}
for split in ['train', 'val', 'test']:
    split_counts[split] = {}
    for species in SELECTED_SPECIES:
        species_dir = DATA_DIR / split / species
        split_counts[split][species] = len([path for path in species_dir.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])

split_table = pd.DataFrame(split_counts).reindex(SELECTED_SPECIES)
split_table['total'] = split_table.sum(axis=1)
display(split_table)
print(split_table.sum().to_dict())
if (split_table[['train', 'val', 'test']] == 0).any().any():
    raise ValueError('At least one class has no samples in a split.')

## 5. Dataset class, transforms, and DataLoaders

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.70, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.12, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class BirdDataset(Dataset):
    def __init__(self, split_dir, class_to_idx, transform):
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.samples = []
        split_dir = Path(split_dir)
        for species, label in class_to_idx.items():
            species_dir = split_dir / species
            for path in sorted(species_dir.iterdir()):
                if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    self.samples.append((path, label))
        if not self.samples:
            raise ValueError(f'No supported images found in {split_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        with Image.open(image_path) as image:
            image = image.convert('RGB')
        return self.transform(image), label

train_dataset = BirdDataset(DATA_DIR / 'train', class_to_idx, train_transform)
val_dataset = BirdDataset(DATA_DIR / 'val', class_to_idx, eval_transform)
test_dataset = BirdDataset(DATA_DIR / 'test', class_to_idx, eval_transform)

generator = torch.Generator().manual_seed(SEED)
loader_options = {'num_workers': NUM_WORKERS, 'pin_memory': device.type == 'cuda'}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator, **loader_options)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

batch_images, batch_labels = next(iter(train_loader))
print(f'Train / validation / test: {len(train_dataset)} / {len(val_dataset)} / {len(test_dataset)}')
print(f'Batch images: {tuple(batch_images.shape)}')
print(f'Batch labels: {tuple(batch_labels.shape)}')

## 6. ResNet-18 model and training helpers

In [ ]:
def build_model(num_classes):
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    for parameter in model.parameters():
        parameter.requires_grad = False
    model.fc = nn.Sequential(
        nn.Dropout(p=DROPOUT),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model, weights

def set_batchnorm_eval(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    set_batchnorm_eval(model)
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    predictions = []
    targets = []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * labels.size(0)
        predicted = logits.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        predictions.extend(predicted.cpu().tolist())
        targets.extend(labels.cpu().tolist())
    return running_loss / total, correct / total, np.asarray(predictions), np.asarray(targets)

model, pretrained_weights = build_model(len(SELECTED_SPECIES))
model = model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print(model)
print(f'Pretrained weights: {pretrained_weights}')
print(f'Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}')

## 7. Train and save the best checkpoint

In [ ]:
optimizer = AdamW(model.fc.parameters(), lr=LEARNING_RATE_HEAD, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=2, min_lr=1e-6)
history = []
best_val_loss = float('inf')
best_epoch = 0
bad_epochs = 0

for epoch in range(1, MAX_EPOCHS + 1):
    if epoch == HEAD_ONLY_EPOCHS + 1:
        for name, parameter in model.named_parameters():
            parameter.requires_grad = name.startswith('layer4.') or name.startswith('fc.')
        optimizer = AdamW([
            {'params': model.layer4.parameters(), 'lr': LEARNING_RATE_BACKBONE},
            {'params': model.fc.parameters(), 'lr': LEARNING_RATE_CLASSIFIER}
        ], weight_decay=WEIGHT_DECAY)
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=2, min_lr=1e-6)
        print('Fine-tuning ResNet layer4 and the classifier head.')

    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(val_loss)
    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy,
        'learning_rate': optimizer.param_groups[0]['lr']
    }
    history.append(row)
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | train loss {train_loss:.4f}, accuracy {train_accuracy:.4f} | val loss {val_loss:.4f}, accuracy {val_accuracy:.4f}')

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_epoch = epoch
        bad_epochs = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': SELECTED_SPECIES,
            'class_to_idx': class_to_idx,
            'epoch': epoch,
            'validation_loss': val_loss,
            'validation_accuracy': val_accuracy,
            'image_size': IMAGE_SIZE,
            'dropout': DROPOUT
        }, CHECKPOINT_PATH)
    else:
        bad_epochs += 1

    if epoch >= HEAD_ONLY_EPOCHS + 1 and bad_epochs >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)
print(f'Best validation-loss checkpoint: epoch {best_epoch}, loss {best_val_loss:.4f}')
print(f'Checkpoint saved at: {CHECKPOINT_PATH}')

## 8. Plot training and validation learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df['epoch'], history_df['train_loss'], marker='o', label='Training')
axes[0].plot(history_df['epoch'], history_df['val_loss'], marker='o', label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].plot(history_df['epoch'], history_df['train_accuracy'], marker='o', label='Training')
axes[1].plot(history_df['epoch'], history_df['val_accuracy'], marker='o', label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

final_row = history_df.iloc[-1]
accuracy_gap = final_row['train_accuracy'] - final_row['val_accuracy']
loss_gap = final_row['val_loss'] - final_row['train_loss']
if accuracy_gap > 0.10 and loss_gap > 0:
    print('The final gap is consistent with overfitting. Increase augmentation or regularization, or stop earlier.')
elif final_row['train_accuracy'] < 0.70 and final_row['val_accuracy'] < 0.70:
    print('Both accuracies are low, which is consistent with underfitting or insufficient training.')
else:
    print('The final training and validation curves are reasonably close under the simple diagnostic thresholds.')
print(f'Final train-validation accuracy gap: {accuracy_gap:.4f}')
print(f'Final validation-training loss gap: {loss_gap:.4f}')

## 9. Test evaluation and confusion matrix

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if checkpoint['class_names'] != SELECTED_SPECIES or checkpoint['class_to_idx'] != class_to_idx:
    raise ValueError('The saved checkpoint class mapping does not match the DataLoader class mapping.')
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_accuracy, y_pred, y_true = evaluate(model, test_loader, criterion)
metrics = {
    'test_dataset_size': int(len(y_true)),
    'best_epoch_by_validation_loss': int(checkpoint['epoch']),
    'test_loss': float(test_loss),
    'accuracy': float(test_accuracy),
    'macro_precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
    'macro_recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
    'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0))
}
with METRICS_PATH.open('w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)
print(json.dumps(metrics, indent=2))
print(classification_report(y_true, y_pred, labels=list(range(len(SELECTED_SPECIES))), target_names=SELECTED_SPECIES, digits=4, zero_division=0))

matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(SELECTED_SPECIES))))
plt.figure(figsize=(12, 10))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=SELECTED_SPECIES, yticklabels=SELECTED_SPECIES, linewidths=0.5, linecolor='#E0E0E0')
plt.title('ResNet-18 Held-Out Test Confusion Matrix')
plt.xlabel('Predicted species')
plt.ylabel('True species')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Classify a new bird image

In [ ]:
def predict_bird(image_path, top_k=3):
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(f'Image not found: {image_path}')
    with Image.open(image_path) as image:
        image = image.convert('RGB')
    input_tensor = eval_transform(image).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probabilities = F.softmax(model(input_tensor), dim=1)[0]
    top_k = max(1, min(int(top_k), len(SELECTED_SPECIES)))
    top_probabilities, top_indices = torch.topk(probabilities, k=top_k)
    return pd.DataFrame({
        'rank': range(1, top_k + 1),
        'species': [idx_to_class[index.item()] for index in top_indices],
        'confidence': top_probabilities.cpu().numpy(),
        'confidence_percent': [f'{value.item() * 100:.2f}%' for value in top_probabilities]
    })

image_path = ''
if image_path:
    display(predict_bird(image_path, top_k=3))
else:
    print('Set image_path to a bird image file and run this cell.')